# PySpark DataFrame DSL — Part 9: Joins and Broadcast Joins

> **Learning goal:** Understand `INNER`, `LEFT`, `RIGHT`, `FULL OUTER`, `CROSS`, `LEFT SEMI`, `LEFT ANTI`, and `BROADCAST` joins in PySpark. Each join starts with a definition and a small example, followed by detailed examples, realistic ETL patterns, performance notes, and practice assignments.

## Table of Contents

1. Setup and sample data
2. Join fundamentals
3. `INNER JOIN`
4. `LEFT JOIN`
5. `RIGHT JOIN`
6. `FULL OUTER JOIN`
7. `CROSS JOIN`
8. `LEFT SEMI JOIN`
9. `LEFT ANTI JOIN`
10. `broadcast()` join hint
11. SQL vs DataFrame DSL
12. Realistic ETL use cases
13. Large-data example
14. Common mistakes and performance
15. Practice assignments
16. Quick reference

---

# 1. Setup and Sample Data

## 1.1 Imports and Spark session

```python
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.functions import broadcast

spark = (
    SparkSession.builder
    .appName("PySpark_Joins")
    .getOrCreate()
)
```

## 1.2 Orders DataFrame

```python
orders_data = [
    (1001, "C101", "Laptop",     65000.0, "COMPLETED"),
    (1002, "C102", "Phone",      32000.0, "COMPLETED"),
    (1003, "C103", "Desk",       12000.0, "PENDING"),
    (1004, "C101", "Headphones",  2500.0, "COMPLETED"),
    (1005, "C104", "Chair",       7500.0, "CANCELLED"),
    (1006, "C105", "Monitor",    18000.0, "COMPLETED"),
    (1007, "C106", "Tablet",     22000.0, "PENDING"),
    (1008, "C107", "Keyboard",    1800.0, "COMPLETED"),
    (1009, "C108", "Phone",      30000.0, "COMPLETED"),
    (1010, "C109", "Desk",       14000.0, "COMPLETED"),
]
df_orders = spark.createDataFrame(
    orders_data,
    ["order_id", "customer_id", "product", "amount", "status"]
)
df_orders.show()
```

## 1.3 Customers DataFrame

Notice that some customers have no orders and one order references a customer not listed in this customer master. This is intentional so unmatched join behavior is visible.

```python
customers_data = [
    ("C101", "Arun",   "Chennai",   "Gold"),
    ("C102", "Priya",  "Bengaluru", "Silver"),
    ("C103", "Rahul",  "Mumbai",    "Gold"),
    ("C104", "Meena",  "Hyderabad", "Bronze"),
    ("C105", "Vikram", "Chennai",   "Silver"),
    ("C106", "Divya",  "Pune",      "Gold"),
    ("C107", "Kavin",  "Chennai",   "Bronze"),
    ("C110", "Sara",   "Delhi",     "Silver"),  # No order in df_orders
]
df_customers = spark.createDataFrame(
    customers_data,
    ["customer_id", "customer_name", "city", "tier"]
)
df_customers.show()
```

Here, `C108` and `C109` have orders but are absent from `df_customers`; `C110` is a customer with no matching order.

## 1.4 Small DataFrame for cross-join examples

```python
df_regions = spark.createDataFrame(
    [("North",), ("South",), ("West",)],
    ["region"]
)

df_channels = spark.createDataFrame(
    [("Web",), ("Store",)],
    ["channel"]
)
```

---

# 2. Join Fundamentals

A **join** combines rows from two DataFrames using a matching condition or a specified combination rule.

```python
df_orders.join(
    df_customers,
    on="customer_id",
    how="inner"
).show()
```

- `df_orders`: left DataFrame.
- `df_customers`: right DataFrame.
- `on`: join key or join condition.
- `how`: join type, such as `"inner"`, `"left"`, `"right"`, `"full"`, `"cross"`, `"left_semi"`, or `"left_anti"`.

## Join key and duplicate columns

When both DataFrames have a column named `customer_id`, joining with `on="customer_id"` generally emits one join-key column. With an explicit condition such as `df_orders.customer_id == df_customers.customer_id`, both key columns may remain in the result, so select/alias columns carefully.

## Join cardinality matters

If the right DataFrame has multiple rows for the same key, one left row can match multiple right rows. This is expected join behavior, but it may accidentally inflate totals if you assume one-to-one matching.

## Null join keys

Standard equality joins do not match null keys to other null keys. If null-safe matching is required, use the appropriate null-safe equality expression deliberately and understand the business meaning.

---

# 3. INNER JOIN

## Definition

An **inner join** returns only rows for which the join condition matches on both sides.

### Simple example

```python
df_orders.join(df_customers, on="customer_id", how="inner").show()
```

Only orders whose `customer_id` exists in `df_customers` are retained. Customers with no order and orders with unknown customer IDs are excluded.

## Syntax

```python
left_df.join(right_df, on="join_key", how="inner")
```

## Examples

### Example 1 — Join using one shared key

```python
df_inner = df_orders.join(
    df_customers,
    on="customer_id",
    how="inner"
)
df_inner.show(truncate=False)
```

**Explanation:** The shared `customer_id` column is used as the matching key. Only matching customer/order pairs appear.

### Example 2 — Select useful columns after joining

```python
df_orders.join(df_customers, "customer_id", "inner").select(
    "order_id", "customer_id", "customer_name", "city", "amount"
).show()
```

**Explanation:** Selecting columns after the join makes the output schema intentional and easier to use downstream.

### Example 3 — Join with an explicit condition and aliases

```python
o = df_orders.alias("o")
c = df_customers.alias("c")

df_inner_explicit = o.join(
    c,
    F.col("o.customer_id") == F.col("c.customer_id"),
    "inner"
).select(
    F.col("o.order_id"),
    F.col("o.customer_id"),
    F.col("c.customer_name"),
    F.col("o.amount")
)
df_inner_explicit.show()
```

**Explanation:** Aliases disambiguate columns when both sides contain similarly named fields. With an explicit condition, choose the output columns explicitly.

### Example 4 — Join with more than one key

```python
df_orders.alias("o").join(
    df_customers.alias("c"),
    (F.col("o.customer_id") == F.col("c.customer_id")) &
    (F.col("o.status") == F.lit("COMPLETED")),
    "inner"
).select("order_id", "customer_id", "customer_name", "status").show()
```

**Explanation:** This condition matches the customer key and restricts output to completed orders. The second condition is a left-side filter embedded in the join condition; for an inner join, filtering completed orders before joining is often clearer.

### Example 5 — SQL inner join

```python
df_orders.createOrReplaceTempView("orders_view")
df_customers.createOrReplaceTempView("customers_view")

spark.sql("""
    SELECT o.order_id, o.customer_id, c.customer_name, c.city, o.amount
    FROM orders_view o
    INNER JOIN customers_view c
      ON o.customer_id = c.customer_id
""").show()
```

**Explanation:** SQL `INNER JOIN` is equivalent to DSL `how="inner"` for the same condition.

---

# 4. LEFT JOIN (LEFT OUTER JOIN)

## Definition

A **left join** returns every row from the left DataFrame and matching rows from the right DataFrame. If there is no right-side match, right-side columns are filled with `NULL`.

### Simple example

```python
df_orders.join(df_customers, on="customer_id", how="left").show()
```

All orders remain, including orders for `C108` and `C109`, which do not exist in the customer master.

## Syntax

```python
left_df.join(right_df, on="key", how="left")
```

`"left_outer"` is also commonly accepted as a synonym.

## Examples

### Example 1 — Keep every order

```python
df_left = df_orders.join(df_customers, "customer_id", "left")
df_left.select("order_id", "customer_id", "customer_name", "amount").show()
```

**Explanation:** All rows from `df_orders` remain. Missing customer attributes become null.

### Example 2 — Find orders with missing customer details

```python
df_orders.join(df_customers, "customer_id", "left").filter(
    F.col("customer_name").isNull()
).select("order_id", "customer_id", "product", "amount").show()
```

**Explanation:** This is a common data-quality check for orders whose customer key is absent from the customer master.

### Example 3 — Add a default label for unmatched rows

```python
df_orders.join(df_customers, "customer_id", "left").withColumn(
    "customer_name_clean",
    F.coalesce(F.col("customer_name"), F.lit("UNKNOWN CUSTOMER"))
).select("order_id", "customer_id", "customer_name_clean").show()
```

**Explanation:** `coalesce()` returns the first non-null expression. The default is for presentation or downstream handling; preserve the original null when auditability requires it.

### Example 4 — Add customer tier and calculate a metric

```python
df_orders.join(df_customers, "customer_id", "left").select(
    "order_id", "customer_id", "tier",
    (F.col("amount") * F.lit(1.18)).alias("amount_with_demo_tax")
).show()
```

**Explanation:** Enriching fact rows with dimension attributes is a standard ETL pattern. The tax rate is illustrative only.

### Example 5 — SQL left join

```python
spark.sql("""
    SELECT o.order_id, o.customer_id, c.customer_name, c.tier
    FROM orders_view o
    LEFT JOIN customers_view c
      ON o.customer_id = c.customer_id
""").show()
```

**Explanation:** SQL `LEFT JOIN` preserves all left-side orders.

---

# 5. RIGHT JOIN (RIGHT OUTER JOIN)

## Definition

A **right join** returns every row from the right DataFrame and matching rows from the left DataFrame. If no left-side match exists, left-side columns become `NULL`.

### Simple example

```python
df_orders.join(df_customers, on="customer_id", how="right").show()
```

This keeps every customer, including `C110`, even though no order matches that customer.

## Syntax

```python
left_df.join(right_df, on="key", how="right")
```

`"right_outer"` is also commonly accepted.

## Examples

### Example 1 — Keep all customers

```python
df_right = df_orders.join(df_customers, "customer_id", "right")
df_right.select("customer_id", "order_id", "customer_name", "city").show()
```

**Explanation:** All customers remain, and `C110` has null order fields.

### Example 2 — Find customers with no orders

```python
df_orders.join(df_customers, "customer_id", "right").filter(
    F.col("order_id").isNull()
).select("customer_id", "customer_name", "city").show()
```

**Explanation:** This finds customer records with no matching order in the orders DataFrame.

### Example 3 — Count orders per customer, including zero

```python
df_orders.join(df_customers, "customer_id", "right").groupBy(
    "customer_id", "customer_name"
).agg(
    F.count("order_id").alias("order_count")
).orderBy("customer_id").show()
```

**Explanation:** `count("order_id")` ignores null order IDs, so the unmatched customer receives zero. Avoid `count("*")` here because the preserved customer row still counts as one joined row.

### Example 4 — Use explicit aliases

```python
o = df_orders.alias("o")
c = df_customers.alias("c")

o.join(
    c,
    F.col("o.customer_id") == F.col("c.customer_id"),
    "right"
).select(
    F.col("c.customer_id"),
    F.col("c.customer_name"),
    F.col("o.order_id"),
    F.col("o.amount")
).show()
```

**Explanation:** Selecting columns by alias avoids ambiguity and makes the preserved side clear.

### Example 5 — SQL right join

```python
spark.sql("""
    SELECT c.customer_id, c.customer_name, o.order_id, o.amount
    FROM orders_view o
    RIGHT JOIN customers_view c
      ON o.customer_id = c.customer_id
""").show()
```

**Explanation:** Right join keeps every row from `customers_view`, the right-side relation.

---

# 6. FULL OUTER JOIN

## Definition

A **full outer join** returns all rows from both sides. Matching keys are combined into one result row per match; unmatched rows from either side remain, with missing-side columns set to `NULL`.

### Simple example

```python
df_orders.join(df_customers, on="customer_id", how="full").show()
```

This preserves orders for unknown customers (`C108`, `C109`) and customers with no orders (`C110`).

## Syntax

```python
left_df.join(right_df, on="key", how="full")
```

Common accepted aliases include `"full_outer"` and `"outer"`.

## Examples

### Example 1 — Preserve both sides

```python
df_full = df_orders.join(df_customers, "customer_id", "full")
df_full.select(
    "customer_id", "order_id", "customer_name", "amount"
).show()
```

**Explanation:** Both unmatched orders and unmatched customers remain in the output.

### Example 2 — Find unmatched rows on either side

```python
df_full.filter(
    F.col("order_id").isNull() | F.col("customer_name").isNull()
).select("customer_id", "order_id", "customer_name").show()
```

**Explanation:** A null order ID indicates a customer with no order; a null customer name indicates an order with no matching customer record. This assumes those fields are reliable non-null identifiers in their source tables.

### Example 3 — Add a match-status label

```python
df_full.withColumn(
    "match_status",
    F.when(F.col("order_id").isNull(), "CUSTOMER_WITHOUT_ORDER")
     .when(F.col("customer_name").isNull(), "ORDER_WITHOUT_CUSTOMER")
     .otherwise("MATCHED")
).select("customer_id", "order_id", "customer_name", "match_status").show()
```

**Explanation:** This makes reconciliation results easier to interpret and report.

### Example 4 — Reconcile distinct keys

```python
df_orders.select("customer_id").distinct().alias("o").join(
    df_customers.select("customer_id").distinct().alias("c"),
    "customer_id",
    "full"
).orderBy("customer_id").show()
```

**Explanation:** This compares key coverage between two datasets without carrying every source column.

### Example 5 — SQL full outer join

```python
spark.sql("""
    SELECT
        COALESCE(o.customer_id, c.customer_id) AS customer_id,
        o.order_id,
        c.customer_name,
        o.amount
    FROM orders_view o
    FULL OUTER JOIN customers_view c
      ON o.customer_id = c.customer_id
""").show()
```

**Explanation:** `COALESCE` selects the first non-null key so the output has one visible key for matched and unmatched rows.

---

# 7. CROSS JOIN

## Definition

A **cross join** returns the Cartesian product: every row on the left is paired with every row on the right. If the left has `m` rows and the right has `n` rows, the output has `m × n` rows.

### Simple example

```python
df_regions.crossJoin(df_channels).show()
```

Three regions × two channels = six rows.

## Syntax

```python
left_df.crossJoin(right_df)
# or
left_df.join(right_df, how="cross")
```

## Examples

### Example 1 — Cartesian product of regions and channels

```python
df_regions.crossJoin(df_channels).show()
```

**Explanation:** Every region is paired with both channels.

### Example 2 — Count the expected number of rows

```python
region_count = df_regions.count()
channel_count = df_channels.count()

print("Expected cross-join rows:", region_count * channel_count)
print("Actual cross-join rows:", df_regions.crossJoin(df_channels).count())
```

**Explanation:** For these small datasets, the expected and actual counts should both be six. `count()` triggers Spark jobs.

### Example 3 — Create a scenario matrix

```python
df_scenarios = spark.createDataFrame(
    [("Low", 0.05), ("Medium", 0.10), ("High", 0.15)],
    ["scenario", "discount_rate"]
)

df_products = spark.createDataFrame(
    [("Laptop", 65000.0), ("Phone", 32000.0)],
    ["product", "price"]
)

df_products.crossJoin(df_scenarios).withColumn(
    "price_after_discount",
    F.col("price") * (F.lit(1.0) - F.col("discount_rate"))
).show()
```

**Explanation:** This evaluates each product under every discount scenario. Two products × three scenarios = six rows.

### Example 4 — Explicit cross join through `join`

```python
df_regions.join(df_channels, how="cross").show()
```

**Explanation:** This is an alternative syntax for a cross join. It does not use a matching key.

### Example 5 — SQL cross join

```python
df_regions.createOrReplaceTempView("regions_view")
df_channels.createOrReplaceTempView("channels_view")

spark.sql("""
    SELECT r.region, c.channel
    FROM regions_view r
    CROSS JOIN channels_view c
""").show()
```

**Explanation:** SQL expresses the Cartesian product explicitly. Use cross joins only when the resulting multiplication is intended and manageable.

---

# 8. LEFT SEMI JOIN

## Definition

A **left semi join** returns only columns from the left DataFrame for rows that have at least one match on the right. It behaves like an existence filter: right-side columns are not included, and duplicate matches on the right do not multiply the left rows.

### Simple example

```python
df_orders.join(df_customers, on="customer_id", how="left_semi").show()
```

This returns orders whose customer ID exists in the customer master.

## Syntax

```python
left_df.join(right_df, on="key", how="left_semi")
```

## Examples

### Example 1 — Orders with known customers

```python
df_known_customer_orders = df_orders.join(
    df_customers, "customer_id", "left_semi"
)
df_known_customer_orders.show()
```

**Explanation:** Only left-side order columns are returned, and only where a matching customer exists.

### Example 2 — Keep only completed orders whose customer exists

```python
df_orders.filter(
    F.col("status") == "COMPLETED"
).join(
    df_customers, "customer_id", "left_semi"
).show()
```

**Explanation:** Filter orders first, then retain rows that have a customer match.

### Example 3 — Semi join with explicit condition

```python
o = df_orders.alias("o")
c = df_customers.alias("c")

o.join(
    c,
    F.col("o.customer_id") == F.col("c.customer_id"),
    "left_semi"
).select("order_id", "customer_id", "amount").show()
```

**Explanation:** Explicit conditions are useful when join keys have different names or need more complex matching. The output remains left-side columns only.

### Example 4 — SQL semi join

```python
spark.sql("""
    SELECT o.*
    FROM orders_view o
    LEFT SEMI JOIN customers_view c
      ON o.customer_id = c.customer_id
""").show()
```

**Explanation:** SQL `LEFT SEMI JOIN` expresses the same existence-filter behavior.

### Example 5 — Semi join versus inner join

```python
df_inner_ids = df_orders.join(
    df_customers, "customer_id", "inner"
).select("order_id", "customer_id")

df_semi_ids = df_orders.join(
    df_customers, "customer_id", "left_semi"
).select("order_id", "customer_id")

print("Inner join row count:", df_inner_ids.count())
print("Semi join row count:", df_semi_ids.count())
```

**Explanation:** With unique customer keys, these counts are equal here. If the right side contains duplicate keys, the inner join can multiply rows; the semi join keeps each matching left row only once.

---

# 9. LEFT ANTI JOIN

## Definition

A **left anti join** returns rows from the left DataFrame that have no matching row on the right. It is the join equivalent of `NOT EXISTS`.

### Simple example

```python
df_orders.join(df_customers, on="customer_id", how="left_anti").show()
```

This returns orders for `C108` and `C109`, whose customer IDs are missing from the customer master.

## Syntax

```python
left_df.join(right_df, on="key", how="left_anti")
```

## Examples

### Example 1 — Find orders with unknown customers

```python
df_unknown_customer_orders = df_orders.join(
    df_customers, "customer_id", "left_anti"
)
df_unknown_customer_orders.show()
```

**Explanation:** This is a common data-quality and referential-integrity check.

### Example 2 — Find customers with no orders

```python
df_customers.join(
    df_orders.select("customer_id").distinct(),
    on="customer_id",
    how="left_anti"
).show()
```

**Explanation:** Here the customer DataFrame is on the left, so the result contains customers with no matching order. The direction matters.

### Example 3 — Find non-completed orders whose customer is not in the master

```python
df_orders.filter(
    F.col("status") != "COMPLETED"
).join(
    df_customers, "customer_id", "left_anti"
).show()
```

**Explanation:** This is a combined filter and anti join. In this sample, unknown customer orders happen to be completed, so this query may return no rows; the logic is still useful for other data.

### Example 4 — SQL anti join

```python
spark.sql("""
    SELECT o.*
    FROM orders_view o
    LEFT ANTI JOIN customers_view c
      ON o.customer_id = c.customer_id
""").show()
```

**Explanation:** SQL returns only order rows for which no customer key matches.

### Example 5 — Anti join for incremental loads

```python
incoming_ids = spark.createDataFrame(
    [(1001,), (1002,), (1011,), (1012,)],
    ["order_id"]
)

existing_ids = df_orders.select("order_id").distinct()

new_ids = incoming_ids.join(
    existing_ids,
    on="order_id",
    how="left_anti"
)

new_ids.show()
```

**Explanation:** This identifies incoming IDs absent from the existing dataset. In a real incremental pipeline, the anti join may be used to identify new records before further validation and write steps. It is not a full substitute for transactional upsert logic when concurrency or updates matter.

---

# 10. BROADCAST JOIN

## Definition

A **broadcast join** is a join strategy in which Spark sends a sufficiently small DataFrame to executors so they can join it locally with partitions of a larger DataFrame. This can avoid shuffling the large side by join key.

In PySpark, `broadcast(df)` marks a DataFrame for broadcast use as a join hint.

### Simple example

```python
df_broadcast_join = df_orders.join(
    broadcast(df_customers),
    on="customer_id",
    how="left"
)
df_broadcast_join.show()
```

Here the small customer dimension is broadcast to workers for joining with the orders dataset.

## Syntax

```python
from pyspark.sql.functions import broadcast

large_df.join(broadcast(small_df), on="key", how="inner")
```

## Parameters and behavior

- `broadcast(df)`: wraps/marks the DataFrame with a broadcast hint.
- `.join(other, on, how)`: performs the join with the specified keys/condition and join type.
- Broadcast joins are especially useful for small dimension tables and larger fact tables.
- A broadcast hint is not a guarantee that a plan is always safe or optimal; Spark's planner and configuration still matter.
- Broadcasting a dataset that is too large can cause memory pressure or executor failures.
- Inspect the physical plan with `explain("formatted")` to confirm the strategy.

## Examples

### Example 1 — Broadcast the customer dimension

```python
df_orders.join(
    broadcast(df_customers),
    on="customer_id",
    how="inner"
).show()
```

**Explanation:** Broadcast is applied to the smaller customer DataFrame; only matching orders are retained by the inner join.

### Example 2 — Broadcast with a left join

```python
df_orders.join(
    broadcast(df_customers),
    on="customer_id",
    how="left"
).select(
    "order_id", "customer_id", "customer_name", "tier", "amount"
).show()
```

**Explanation:** All orders remain, and customer attributes are added where a match exists.

### Example 3 — Broadcast after selecting only needed dimension columns

```python
small_customer_dim = df_customers.select(
    "customer_id", "customer_name", "tier"
)

df_orders.join(
    broadcast(small_customer_dim),
    on="customer_id",
    how="left"
).show()
```

**Explanation:** Reducing columns reduces the amount of data broadcast. Filter and project the dimension appropriately before broadcasting.

### Example 4 — Broadcast a filtered dimension

```python
gold_customers = df_customers.filter(F.col("tier") == "Gold")

df_orders.join(
    broadcast(gold_customers),
    on="customer_id",
    how="inner"
).show()
```

**Explanation:** Only orders belonging to Gold-tier customers are retained. This is a join against a filtered dimension.

### Example 5 — SQL broadcast hint

```python
spark.sql("""
    SELECT /*+ BROADCAST(c) */
        o.order_id, o.customer_id, c.customer_name, c.tier, o.amount
    FROM orders_view o
    LEFT JOIN customers_view c
      ON o.customer_id = c.customer_id
""").show()
```

**Explanation:** Spark SQL supports join hints such as `BROADCAST`. Hints guide the optimizer; verify the actual physical plan.

### Example 6 — Inspect the physical plan

```python
df_broadcast = df_orders.join(
    broadcast(df_customers),
    on="customer_id",
    how="left"
)

df_broadcast.explain("formatted")
```

**What to look for:** Depending on Spark version and query, the plan may show `BroadcastHashJoin` and `BroadcastExchange`. If it does not, check the plan, Spark version, and configuration.

### Example 7 — Compare with a regular join

```python
regular_join = df_orders.join(
    df_customers, "customer_id", "left"
)

broadcast_join = df_orders.join(
    broadcast(df_customers), "customer_id", "left"
)

regular_join.explain("formatted")
broadcast_join.explain("formatted")
```

**Explanation:** Compare the physical plans. Runtime comparisons on a tiny sample are not meaningful; use representative data and a controlled benchmark.

---

# 11. SQL vs DataFrame DSL

Register views:

```python
df_orders.createOrReplaceTempView("orders_view")
df_customers.createOrReplaceTempView("customers_view")
```

## Inner join

**DSL**
```python
df_orders.join(df_customers, "customer_id", "inner")
```

**SQL**
```sql
SELECT *
FROM orders_view o
INNER JOIN customers_view c
  ON o.customer_id = c.customer_id
```

## Left join

**DSL**
```python
df_orders.join(df_customers, "customer_id", "left")
```

**SQL**
```sql
SELECT *
FROM orders_view o
LEFT JOIN customers_view c
  ON o.customer_id = c.customer_id
```

## Right join

**DSL**
```python
df_orders.join(df_customers, "customer_id", "right")
```

**SQL**
```sql
SELECT *
FROM orders_view o
RIGHT JOIN customers_view c
  ON o.customer_id = c.customer_id
```

## Full outer join

**DSL**
```python
df_orders.join(df_customers, "customer_id", "full")
```

**SQL**
```sql
SELECT *
FROM orders_view o
FULL OUTER JOIN customers_view c
  ON o.customer_id = c.customer_id
```

## Cross join

**DSL**
```python
df_regions.crossJoin(df_channels)
```

**SQL**
```sql
SELECT *
FROM regions_view
CROSS JOIN channels_view
```

## Left semi join

**DSL**
```python
df_orders.join(df_customers, "customer_id", "left_semi")
```

**SQL**
```sql
SELECT o.*
FROM orders_view o
LEFT SEMI JOIN customers_view c
  ON o.customer_id = c.customer_id
```

## Left anti join

**DSL**
```python
df_orders.join(df_customers, "customer_id", "left_anti")
```

**SQL**
```sql
SELECT o.*
FROM orders_view o
LEFT ANTI JOIN customers_view c
  ON o.customer_id = c.customer_id
```

## Broadcast join

**DSL**
```python
df_orders.join(broadcast(df_customers), "customer_id", "left")
```

**SQL**
```sql
SELECT /*+ BROADCAST(c) */ o.*, c.customer_name, c.tier
FROM orders_view o
LEFT JOIN customers_view c
  ON o.customer_id = c.customer_id
```

**Key takeaway:** Join semantics are the same across DSL and SQL for equivalent conditions. Use the API your team can maintain, and inspect plans for performance-sensitive joins.

---

# 12. Realistic ETL Use Cases

## Use case 1 — Enrich a fact table with a dimension (left join)

```python
enriched_orders = df_orders.join(
    df_customers.select("customer_id", "customer_name", "city", "tier"),
    on="customer_id",
    how="left"
)
```

Use this when every fact row should remain, even if dimension data is missing. Add a data-quality flag for unmatched keys if needed.

## Use case 2 — Reject or quarantine invalid foreign keys (anti join)

```python
valid_orders = df_orders.join(
    df_customers.select("customer_id").distinct(),
    on="customer_id",
    how="left_semi"
)

invalid_orders = df_orders.join(
    df_customers.select("customer_id").distinct(),
    on="customer_id",
    how="left_anti"
)
```

`valid_orders` has known customer keys; `invalid_orders` has keys absent from the customer reference dataset. In production, write invalid rows to a quarantine path/table with an error reason.

## Use case 3 — Reconcile two source systems (full outer)

```python
source_a = spark.createDataFrame(
    [("K1", 100.0), ("K2", 200.0), ("K3", 300.0)],
    ["business_key", "amount_a"]
)
source_b = spark.createDataFrame(
    [("K1", 100.0), ("K2", 220.0), ("K4", 400.0)],
    ["business_key", "amount_b"]
)

reconciliation = source_a.join(source_b, "business_key", "full").withColumn(
    "reconciliation_status",
    F.when(F.col("amount_a").isNull(), "ONLY_IN_B")
     .when(F.col("amount_b").isNull(), "ONLY_IN_A")
     .when(F.col("amount_a") != F.col("amount_b"), "AMOUNT_MISMATCH")
     .otherwise("MATCHED")
)
reconciliation.show()
```

Use this to identify missing keys and value discrepancies. For floating-point money, production systems should generally use an appropriate decimal type and explicit comparison tolerance/rounding policy.

## Use case 4 — Generate a product/channel matrix (cross join)

```python
products = spark.createDataFrame(
    [("Laptop",), ("Phone",), ("Tablet",)], ["product"]
)
channels = spark.createDataFrame(
    [("Web",), ("Store",)], ["channel"]
)

product_channel_matrix = products.crossJoin(channels)
product_channel_matrix.show()
```

Use when every combination is intentionally required, for example a planning matrix. Confirm the row multiplication is manageable.

## Use case 5 — Broadcast a small dimension

```python
fact_orders = df_orders
small_dim = df_customers.select("customer_id", "tier")

fact_enriched = fact_orders.join(
    broadcast(small_dim),
    on="customer_id",
    how="left"
)
```

Useful when a small dimension can fit comfortably in executor memory. Measure and inspect plans rather than broadcasting every table by default.

---

# 13. Large-Data Example

Use Spark-native generation rather than constructing a million Python records on the driver.

```python
large_orders = (
    spark.range(0, 1_000_000)
    .withColumnRenamed("id", "order_id")
    .withColumn("customer_id", F.concat(F.lit("C"), (F.col("order_id") % 100_000).cast("string")))
    .withColumn("amount", (F.col("order_id") % 10000 + 100).cast("double"))
    .withColumn("status", F.when(F.col("order_id") % 5 == 0, "PENDING").otherwise("COMPLETED"))
)

large_customers = (
    spark.range(0, 100_000)
    .withColumnRenamed("id", "customer_num")
    .withColumn("customer_id", F.concat(F.lit("C"), F.col("customer_num").cast("string")))
    .withColumn("customer_tier", F.when(F.col("customer_num") % 3 == 0, "Gold").otherwise("Standard"))
    .select("customer_id", "customer_tier")
)
```

## Example A — Large inner join

```python
large_inner = large_orders.join(
    large_customers, "customer_id", "inner"
)
large_inner.groupBy("customer_tier").agg(
    F.count("*").alias("rows"),
    F.sum("amount").alias("total_amount")
).show()
```

## Example B — Broadcast the dimension (only if it fits)

```python
# 100,000 rows may or may not be small enough depending on row width,
# cluster memory, and configuration. Inspect size and test in your environment.
large_broadcast_join = large_orders.join(
    broadcast(large_customers), "customer_id", "inner"
)
large_broadcast_join.explain("formatted")
```

Do not assume that a DataFrame is safe to broadcast based only on row count. Estimate its serialized size, check executor memory, and inspect the actual plan.

## Example C — Find unmatched fact keys

```python
unmatched_orders = large_orders.join(
    large_customers.select("customer_id"),
    "customer_id",
    "left_anti"
)
print("Unmatched count:", unmatched_orders.count())
```

In this synthetic example, all generated order keys should match the generated customer keys. Change the key-generation logic to test missing-key scenarios.

---

# 14. Common Mistakes and Performance Notes

## Mistake 1 — Using the wrong join direction

`left_anti` returns rows from the left side only. To find customers without orders, customers must be on the left:

```python
df_customers.join(df_orders.select("customer_id").distinct(),
                  "customer_id", "left_anti")
```

## Mistake 2 — Duplicate keys multiply rows

If the right side has three matching rows for one key, a left row can appear three times in an inner/left join. Check dimension uniqueness:

```python
df_customers.groupBy("customer_id").count().filter(F.col("count") > 1).show()
```

Deduplicate only according to a valid business rule; do not drop duplicates blindly.

## Mistake 3 — Counting rows incorrectly after an outer join

For customers with no order, `count("*")` counts the preserved joined row, but `count("order_id")` returns zero because the order ID is null.

## Mistake 4 — Selecting ambiguous columns

When both DataFrames have a `status`, `city`, or key column, alias the DataFrames and select qualified columns explicitly.

## Mistake 5 — Broadcasting a large DataFrame

Broadcasting an oversized dataset can exhaust executor memory. Broadcast only when the smaller side is genuinely small enough.

## Mistake 6 — Assuming every join uses a shuffle

Spark may choose broadcast hash join, sort-merge join, shuffled hash join, or another strategy based on the query, configuration, and data statistics. Use `explain("formatted")`.

## Performance checklist

- Filter rows and select columns before joining when logically correct.
- Ensure join-key types are compatible.
- Check duplicate keys and expected join cardinality.
- Broadcast only suitable small dimensions.
- Watch skewed keys, which can create uneven task sizes.
- Avoid cross joins unless Cartesian combinations are required.
- Use semi joins for existence filtering and anti joins for non-matching records.
- Inspect the physical plan and measure with representative data.

---

# 15. Practice Assignments

Use `df_orders` and `df_customers` created in Section 1.

## Beginner

1. Perform an inner join and display order ID, customer name, city, and amount.
2. Perform a left join and identify orders without customer details.
3. Perform a right join and show customers with no orders.
4. Perform a full outer join and label matched versus unmatched rows.
5. Create a cross join between three regions and two sales channels; confirm the row count.

## Intermediate

6. Use a left semi join to find orders with valid customer keys.
7. Use a left anti join to find orders with missing customer keys.
8. Use a left anti join with customers on the left to find customers without orders.
9. Use a full outer join to reconcile two sources with different amounts.
10. Write an inner and left join in both DSL and SQL.

## Advanced

11. Make the customer dimension contain duplicate keys. Demonstrate how the inner join row count changes, then propose a valid deduplication rule.
12. Use `broadcast()` with a small dimension and inspect the formatted plan.
13. Compare a regular join with a broadcast-hinted join on representative data; explain why a tiny sample is not a useful benchmark.
14. Build a valid-orders DataFrame with semi join and an invalid-orders DataFrame with anti join, then describe how you would quarantine invalid records.
15. Create a scenario matrix using a cross join and explain how to estimate its output size before running it.

---

# 16. Quick Reference

| Join type | PySpark `how` value | What it returns |
|---|---|---|
| Inner | `"inner"` | Only matching row pairs |
| Left outer | `"left"` / `"left_outer"` | All left rows, matching right data, nulls for missing right matches |
| Right outer | `"right"` / `"right_outer"` | All right rows, matching left data, nulls for missing left matches |
| Full outer | `"full"` / `"full_outer"` / `"outer"` | All rows from both sides |
| Cross | `crossJoin()` or `"cross"` | Every left/right row combination |
| Left semi | `"left_semi"` | Left rows that have a match; left columns only |
| Left anti | `"left_anti"` | Left rows that have no match; left columns only |
| Broadcast | `broadcast(small_df)` hint | Not a separate join type; encourages a broadcast join strategy |

## Final Takeaway

- Use **inner** for matched records only.
- Use **left** to preserve all left-side records while enriching them.
- Use **right** to preserve all right-side records; many pipelines rewrite it as a left join by swapping inputs for readability.
- Use **full outer** for reconciliation across both datasets.
- Use **cross** only when every combination is intentional.
- Use **left semi** for “does a match exist?” and **left anti** for “no match exists.”
- Use **broadcast** as a performance hint for a suitably small relation—not as a replacement for understanding join semantics.
